In [ ]:
import shap

# Extract the raw XGBoost model from the tuned pipeline
best_model  = best_xgb.named_steps['model']
best_scaler = best_xgb.named_steps['scaler']

X_holdout_scaled = best_scaler.transform(X_holdout)
X_holdout_df     = pd.DataFrame(X_holdout_scaled, columns=feature_cols)

# Build SHAP explainer
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_holdout_df)

# --- Global explanation: what does the model rely on overall? ---
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_holdout_df, feature_names=feature_cols,
                  plot_type="bar", show=False)
plt.title("Global Feature Importance (mean |SHAP value|)")
plt.tight_layout(); plt.show()

# --- Detailed SHAP plot: how does each feature VALUE affect predictions? ---
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_holdout_df, feature_names=feature_cols, show=False)
plt.title("SHAP Summary: Feature Values vs Prediction Impact")
plt.tight_layout(); plt.show()
python
# --- Local explanation: why did the model make THIS specific prediction? ---
# Pick an interesting day — e.g. the largest wrong prediction
wrong_idx = np.where(tuned_preds != y_holdout.values)[0]
sample_idx = wrong_idx[0]  # First wrong prediction

print(f"\nExplaining prediction for: {X_holdout.index[sample_idx].date()}")
print(f"Model predicted: {'Up' if tuned_preds[sample_idx] == 1 else 'Down'}")
print(f"Actual outcome:  {'Up' if y_holdout.iloc[sample_idx] == 1 else 'Down'}")
print(f"Confidence (proba): {tuned_proba[sample_idx]:.3f}")

shap.force_plot(
    explainer.expected_value,
    shap_values[sample_idx],
    X_holdout_df.iloc[sample_idx],
    feature_names=feature_cols,
    matplotlib=True,
    show=True
)


In [ ]:
python
# --- Production-ready prediction pipeline ---
# This is how you'd use the model in a live system.

class QuantPredictionPipeline:
    """
    A production-ready wrapper around our trained ML model.
    Given a stock ticker, it downloads the latest data,
    engineers features, and returns a trading signal.
    """
    def __init__(self, model, scaler, feature_cols, spy_ticker="SPY"):
        self.model        = model
        self.scaler       = scaler
        self.feature_cols = feature_cols
        self.spy_ticker   = spy_ticker

    def get_latest_signal(self, ticker, lookback_days=300):
        """
        Downloads fresh data and returns:
        - signal: 1 (go long) or 0 (stay out)
        - confidence: probability of an up day
        - top_drivers: the 3 features pushing the prediction
        """
        # Download fresh data
        end_date   = pd.Timestamp.today()
        start_date = end_date - pd.Timedelta(days=lookback_days)

        raw    = yf.download(ticker,     start=start_date, end=end_date,
                             auto_adjust=True, progress=False)
        spy    = yf.download(self.spy_ticker, start=start_date, end=end_date,
                             auto_adjust=True, progress=False)

        if len(raw) < 60:
            raise ValueError(f"Not enough data for {ticker}")

        # Engineer features
        featured = build_features(raw, spy)

        # Get the most recent complete row
        latest = featured[self.feature_cols].dropna().iloc[[-1]]

        # Scale and predict
        latest_scaled = self.scaler.transform(latest)
        proba  = self.model.predict_proba(latest_scaled)[0, 1]
        signal = int(proba > 0.5)

        # SHAP explanation for this prediction
        exp          = shap.TreeExplainer(self.model)
        shap_vals    = exp.shap_values(pd.DataFrame(latest_scaled, columns=self.feature_cols))
        top_idx      = np.argsort(np.abs(shap_vals[0]))[::-1][:3]
        top_drivers  = [(self.feature_cols[i], round(shap_vals[0][i], 4)) for i in top_idx]

        return {
            'ticker':     ticker,
            'date':       latest.index[0].date(),
            'signal':     signal,
            'confidence': round(proba, 4),
            'action':     "BUY / HOLD LONG" if signal == 1 else "STAY OUT / FLAT",
            'top_drivers': top_drivers
        }


# Instantiate and run the pipeline
pipeline = QuantPredictionPipeline(
    model        = best_model,
    scaler       = best_scaler,
    feature_cols = feature_cols
)

result = pipeline.get_latest_signal("AAPL")

print("\n" + "="*50)
print(f"  LIVE PREDICTION for {result['ticker']}")
print("="*50)
print(f"  Date:       {result['date']}")
print(f"  Signal:     {result['signal']}")
print(f"  Action:     {result['action']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"\n  Top prediction drivers:")
for feat, shap_val in result['top_drivers']:
    direction = "▲ pushes UP" if shap_val > 0 else "▼ pushes DOWN"
    print(f"    {feat:25s}  {direction}  (SHAP: {shap_val:+.4f})")
print("="*50)
What you've now built
You have a complete, professional-grade quantitative ML system:



In [ ]:

EDA that understands the statistical character of the data before touching a model
30+ engineered features covering momentum, trend, volatility, volume, and market context
4 models trained with proper walk-forward cross-validation (no data leakage)
Hyperparameter tuning using randomized search over time-series folds
A backtesting engine that calculates Sharpe, Sortino, Calmar, max drawdown, win rate, and profit factor — including transaction costs
SHAP explainability at both the global level (what the model relies on) and local level (why it made this specific call)
A production pipeline class you can call with any ticker to get a live signal
The natural next step from here is multi-asset portfolio construction — running this model on 20-50 stocks simultaneously and sizing positions based on confidence, or adding regime filtering so the model only trades when market conditions favor its edge. Tell me which direction you want to take it.





Claude is AI and can make mistakes. Please double-check responses.

